# CellPhe

<!-- badges: start -->
<a href="https://zenodo.org/badge/latestdoi/449769672"><img src="https://zenodo.org/badge/449769672.svg" alt="DOI"></a>
<!-- badges: end -->

CellPhe provides functions to phenotype cells from time-lapse videos and accompanies the paper:\
Wiggins, L., Lord, A., Murphy, K.L. et al.\
The CellPhe toolkit for cell phenotyping using time-lapse imaging and pattern recognition.\
Nat Commun 14, 1854 (2023).\
https://doi.org/10.1038/s41467-023-37447-3

The Python package is a port of the [original R implementation](https://github.com/uoy-research/CellPhe).

## Installation

You can install the latest version of CellPhe from
[PyPi](https://pypi.org/project/cellphe/) with:

```
pip install cellphe
```

The default installation provides access to the core phenotyping functionality, but if you would also like to segment and track your images, the full installation will need to be installed as below. Segmentation and tracking have large dependencies and so are not included by default.

```
pip install cellphe[full]
```

## Example

An example dataset to demonstrate CellPhe’s capabilities is hosted on [Dryad](https://doi.org/10.5061/dryad.4xgxd25f0) in the archive `example_data.zip` and comprises 3 parts:

-   The time-lapse stills as TIFF images (`05062019_B3_3_imagedata`)
-   Existing pre-extracted features from PhaseFocus Livecyte.
    (`05062019_B3_3_Phase-FullFeatureTable.csv`)
-   Region-of-interest (ROI) boundaries already demarked in ImageJ
    format (`05062019_B3_3_Phase`)

These should be extracted into a suitable location (this guide assumes they have been extracted into `data`) before proceeding
with the rest of the tutorial.

The first step in the CellPhe workflow is to prepare a dataframe containing
metadata identifying the tracked cells across all the frames, along with any
pre-existing attributes. The segmenting and tracking can be performed within
CellPhe, or pre-segmented and tracked data from two widely used software
(PhaseFocus Livecyte & Trackmate) can be directly imported.




### Segmenting and tracking

**NB: Please ensure that you have installed the full version of CellPhe as shown above before segmenting or tracking.**
This feature is still experimental, please report any bugs at the [issue tracker](https://github.com/uoy-research/CellPhePy/issues).


CellPhe provides 2 functions to segment and track an image sequence:

  - `segment_images`: Segments images using Cellpose
  - `track_images`: Uses the ImageJ plugin TrackMate to track cells between frames without requiring ImageJ to be installed


In [1]:
from cellphe import segment_images, track_images

`segment_images` takes 4 arguments:

  - the path to the directory where the images are stored (where the folder `05062019_B3_3_imagedata` was extracted to)
  - a path to an output folder where the resultant Cellpose masks will be saved
  - parameters for the CellPose model instantiation, including the model type (defaults to `cyto3`).
  - parameters for the CellPose `eval` function which governs the segmentation

For the latter 2, refer to the [CellPose docs](https://cellpose.readthedocs.io/en/latest/models.html) for a full list of options.

Segmentation can take several minutes depending on the number of images and their resolution.

In [3]:
segment_images("data/05062019_B3_3_imagedata", "data/masks")

Unable to load cellpose. Ensure that the full version of cellphe was installed with `pip install cellphe[full]`.


Confirm that the `masks` directory has been created and populated with TIFs containing cell masks.
If it has, then you are ready to track the cells.
`track_images` takes at minimum 3 arguments: 

  - the location of the masks created by `segment_images`
  - the filename to save the output metadata to
  - a filename for the output ROI zip

Optionally you can also change the tracking options - by default the Simple LAP method is employed - with the `tracker` and `tracker_settings` arguments.


In [12]:
track_images("data/masks", "data/tracked.csv", "data/rois.zip", tracker="SimpleSparseLAP")

Computing spot features over 1 frame simultaneously and allocating 8 threads per frame.
Calculating 0 spots features...

Computation done in 1 ms.
Computing track features:
  - Branching analyzer in 0 ms.
  - Track duration in 0 ms.
  - Track index in 0 ms.
  - Track location in 0 ms.
  - Track speed in 0 ms.
  - Track quality in 0 ms.
  - Track motility analysis in 0 ms.
Computation done in 0 ms.
Starting detection process using 8 threads.
Detection processes 8 frames simultaneously and allocates 1 thread per frame.
Detection...
Found 22958 spots.

Starting initial filtering process.
Computing spot features over 8 frames simultaneously and allocating 1 thread per frame.
Calculating 22958 spots features...

Computation done in 4882 ms.
Starting spot filtering process.
Starting tracking process.
Frame to frame linking...

Creating the segment linking cost matrix...
Creating the main cost matrix...
Completing the cost matrix...
Solving the cost matrix...

Creating links...


Computing ed

Confirm that the `tracked.csv` and `rois.zip` files were created.
These outputs can now be loaded into CellPhe.

### Importing pre-segmented and tracked data

Once a metadata file (CSV format) and a zip of ROIs are available, either directly output from external software (PhaseFocus Livecyte or TrackMate in ImageJ), or from within CellPhe as in the previous section, they can be read into CellPhe.
The `import_data` function accepts metadata files from one of these sources and converts it into a standard format.
It takes 3 arguments: the metadata file path, the source, and the minimum number of frames that a cell must be tracked for to be retained in the dataset (optional).

For example, the dataset that was segmented and tracked in the previous section can be imported as:

In [4]:
from cellphe import import_data

In [5]:
feature_table = import_data("data/tracked.csv", "Trackmate_auto", 50)

In [6]:
feature_table.head()

,FrameID,CellID,ROI_filename
30,1,1,001-001
75,2,1,002-001
116,3,1,003-001
184,4,1,004-001
207,5,1,005-001


Alternatively, the example below creates the metadata dataframe from the supplied PhaseFocus segmentation and tracking, only including cells that were tracked for at least 50 frames.

In [7]:
input_feature_table = "data/05062019_B3_3_Phase-FullFeatureTable.csv"
feature_table_phase = import_data(input_feature_table, "Phase", 50)
feature_table_phase.head()

,FrameID,CellID,ROI_filename,Volume,Sphericity
3,1,2,1-2,603.048401,0.852517
4,2,2,2-2,618.647583,0.784101
5,3,2,3-2,652.385071,0.805910
6,4,2,4-2,659.943665,0.802412
7,5,2,5-2,648.213989,0.769735


If a segmented and tracked dataset is available from a different source then it can still be used in CellPhe provided that it can be loaded into a `pandas.DataFrame` containing:

  - Each row corresponding to a cell tracked in a specific frame
  - A column `FrameID` (integer) denoted the frame number in chronological order
  - A column `CellID` (integer) identifying the cell
  - A column `ROI_filename` (string) denoting the filename (without extension) of the corresponding ROI file, not including the full path

Additional columns providing cell features can be included and will be retained and incorporated into the CellPhe analysis.
The PhaseFocus dataset keeps the volume and sphericity features, for example.

### Generating cell features

In addition to any pre-calculated features, the `cell_features()`
function generates 74 descriptive features for each cell on every frame
using the frame images and pre-generated cell boundaries, based on size,
shape, texture, and the local cell density. The output is a dataframe
comprising the `FrameID`, `CellID`, and `ROI_filename` columns
from the feature table input, the 74 features as columns,
and any additional features that may be present (such as from `import_data()`)
in further columns.

`cell_features()` takes as arguments the feature table, the archive where ROIs are saved, the folder where the images are, and the framerate.
It expects images to be saved with a filename ending with the frame id just before the file extension. The file extension can be `.tif`, `.tiff`, or the `ome.tif` and `.ome.tiff` equivalents. 
The frame id can be zero-padded or not.
`myexperiment-1.tif`, `myexperiment_1.tiff`, `2.ome.tif` are all valid names.

ROI files are named according to the `ROI_filename` column with a `.roi` extension.

The example below uses the PhaseFocus ROIs, but the ones generated using TrackMate just before can be used with the corresponding feature table.
There are 74 features generated during this step, which added to the 3 identifiers (`FrameID`, `CellID`, `ROI_filename`) and 2 PhaseFocus features (`Volume`, `Sphericity`) results in 79 columns.

In [8]:
from cellphe import cell_features
roi_archive = "data/05062019_B3_3_Phase.zip"
image_folder = "data/05062019_B3_3_imagedata"
new_features = cell_features(feature_table_phase, roi_archive, image_folder, framerate=0.0028)

In [9]:
new_features.head()

,FrameID,CellID,ROI_filename,Volume,Sphericity,Dis,Trac,D2T,Vel,Rad,...,IQ3,IQ4,IQ5,IQ6,IQ7,IQ8,IQ9,x,y,dens
0,1,2,1-2,603.048401,0.852517,0.000000,0.000000,0.000000,0.000000,8.681769,...,1.317259,1.216603,1.102974,0.989025,0.835997,0.749949,0.641646,88.903846,518.057692,0.072316
1,2,2,2-2,618.647583,0.784101,2.353424,2.353424,1.000000,0.006590,8.887084,...,1.342967,1.228169,1.116847,1.009470,0.872957,0.688360,0.456557,89.949153,515.949153,0.071690
2,3,2,3-2,652.385071,0.805910,1.602057,3.189590,0.502277,0.002341,9.293634,...,1.449308,1.341675,1.218744,1.110506,0.960103,0.790081,0.576323,89.872727,516.781818,0.070477
3,4,2,4-2,659.943665,0.802412,2.951481,4.761765,0.619829,0.004402,9.430428,...,1.468752,1.366378,1.245534,1.118893,1.032372,0.845990,0.653296,89.709091,515.218182,0.073441
4,5,2,5-2,648.213989,0.769735,3.721509,5.538301,0.671959,0.002174,9.576764,...,1.484971,1.378171,1.252418,1.128258,1.019737,0.779663,0.534423,89.810345,514.448276,0.074045


### Generating time-series features

The next step is to calculate features that incorporate the time-dimension.
This is done with the `time_series_features` function, which accepts a dataframe with the cell-level features as output earlier from `cell_features`.

Variables are calculated from the time series providing both
summary statistics and indicators of time-series behaviour at different
levels of detail obtained via wavelet analysis. 15 summary scores are
calculated for each feature, in addition to the cell trajectory, thereby
resulting in a default output of 1081 features (15x72 + 1). 
With the 2 PhaseFocus features as well, this increases to 1111.
The output is a dataframe with the first column being the
`CellID` used previously.

In [10]:
from cellphe import time_series_features
ts_variables = time_series_features(new_features)

In [11]:
ts_variables.head()

,CellID,A2B_mean,A2B_std,A2B_skew,Area_mean,Area_std,Area_skew,Box_mean,Box_std,Box_skew,...,poly4_l1_asc,poly4_l1_des,poly4_l1_max,poly4_l2_asc,poly4_l2_des,poly4_l2_max,poly4_l3_asc,poly4_l3_des,poly4_l3_max,trajArea
0,2,0.095047,0.008466,-1.277404,346.259875,56.441240,1.206047,1.338842,0.093507,1.926193,...,2.369861,-2.460841,47.259019,2.572310,-1.611360,29.186845,1.753271,-1.919305,12.797446,0.623323
1,3,0.092142,0.011569,-1.615687,688.148936,216.608400,1.393530,1.351822,0.113257,1.289532,...,3.554182,-2.072767,35.853083,2.500715,-1.495602,21.405508,1.595877,-3.060472,11.730594,83.530671
2,5,0.092124,0.009241,-1.319829,487.855649,73.061202,-0.378465,1.345572,0.079226,1.022087,...,2.783071,-3.223079,49.745435,2.567780,-2.184085,19.802078,1.555677,-3.011515,30.409920,5.101182
3,6,0.085164,0.010464,0.375708,548.186957,146.269491,-0.317095,1.291496,0.065032,0.740871,...,5.495591,-3.794187,87.033024,4.200457,-3.292587,50.855024,2.239058,-4.402756,8.215694,2.175653
4,7,0.085734,0.011498,-0.359607,728.992958,245.370656,1.311465,1.391985,0.115098,0.755004,...,4.625511,-3.817404,31.558305,3.710931,-5.180719,21.766337,3.796728,-3.213180,14.422016,12.269081
